# Kepler Classification

In [12]:
import pandas as pd

# 1. Load your CSV file
df = pd.read_csv("../data/space_data.csv")

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset dimensions
print("Dataset shape:", df.shape)

# Display first five observations
display(df.head())

# Target class distribution
print("\nTarget class distribution:")
print(df["koi_disposition"].value_counts())

# Check column types
print("\nColumn data types:")
print(df.dtypes.value_counts())

Dataset shape: (9564, 50)


,rowid,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,...,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag
0,1,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,...,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
1,2,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,...,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
2,3,10811496,K00753.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,...,-176.0,4.544,0.044,-0.176,0.868,0.233,-0.078,297.00482,48.134129,15.436
3,4,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,...,-174.0,4.564,0.053,-0.168,0.791,0.201,-0.067,285.53461,48.285210,15.597
4,5,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,...,-211.0,4.438,0.070,-0.210,1.046,0.334,-0.133,288.75488,48.226200,15.509



Target class distribution:
koi_disposition
FALSE POSITIVE    5023
CONFIRMED         2293
CANDIDATE         2248
Name: count, dtype: int64

Column data types:
float64    39
int64       6
str         5
Name: count, dtype: int64


In [14]:
# 1. Identify missing values
missing = df.isnull().sum()

missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": (missing / len(df) * 100).round(2)
})

display(
    missing_report[missing_report["Missing Count"] > 0]
    .sort_values("Missing %", ascending=False)
)

# 2. Identify categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns

print("\nCategorical columns:")
print(categorical_cols.tolist())

# 3. Check duplicates
print("\nDuplicate rows:", df.duplicated().sum())

# 4. Check unique stars
print("Unique Kepler IDs:", df["kepid"].nunique())

,Missing Count,Missing %
koi_teq_err1,9564,100.00
koi_teq_err2,9564,100.00
kepler_name,7270,76.01
koi_score,1510,15.79
koi_steff_err2,483,5.05
koi_slogg_err1,468,4.89
koi_slogg_err2,468,4.89
koi_srad_err2,468,4.89
koi_steff_err1,468,4.89
koi_srad_err1,468,4.89



Categorical columns:
['kepoi_name', 'kepler_name', 'koi_disposition', 'koi_pdisposition', 'koi_tce_delivname']

Duplicate rows: 0
Unique Kepler IDs: 8214


/tmp/ipykernel_33800/22574644.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns


In [15]:
# Define target
target = "koi_disposition"

# Define columns to exclude
drop_columns = [
    "koi_disposition",
    "kepid",
    "kepoi_name",
    "kepler_name",
    "koi_pdisposition",
    "koi_score",
    "koi_fpflag_nt",
    "koi_fpflag_ss",
    "koi_fpflag_co",
    "koi_fpflag_ec",
    "koi_tce_delivname",
    "koi_teq_err1",
    "koi_teq_err2"
]

# Separate features and target
X = df.drop(columns=drop_columns)
y = df[target]

# Preserve star identifiers for group-aware splitting
groups = df["kepid"]

# Check remaining features
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nRemaining feature types:")
print(X.dtypes.value_counts())

print("\nTarget classes:")
print(y.value_counts())

print("\nMissing values remaining:")
print(X.isnull().sum().sum())

# Validate features
assert X.select_dtypes(exclude="number").empty, \
    "Non-numeric features remain!"

assert not y.isnull().any(), \
    "Target contains missing values!"

print("\nFeature selection successful!")

Feature matrix shape: (9564, 37)
Target shape: (9564,)

Remaining feature types:
float64    36
int64       1
Name: count, dtype: int64

Target classes:
koi_disposition
FALSE POSITIVE    5023
CONFIRMED         2293
CANDIDATE         2248
Name: count, dtype: int64

Missing values remaining:
12303

Feature selection successful!


In [16]:
from sklearn.model_selection import GroupShuffleSplit

# Create group-aware splitter
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

# Generate train/test indices
train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Split features and target
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Preserve group identifiers
train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

# Display dataset sizes
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).round(3))

# Check for group leakage
overlapping_stars = set(train_groups).intersection(
    set(test_groups)
)

print("\nOverlapping stars:", len(overlapping_stars))

assert len(overlapping_stars) == 0, \
    "Data leakage detected!"

print("\nGroup-aware splitting successful!")

Training observations: 7662
Testing observations: 1902

Training class distribution:
koi_disposition
FALSE POSITIVE    0.522
CONFIRMED         0.241
CANDIDATE         0.236
Name: proportion, dtype: float64

Testing class distribution:
koi_disposition
FALSE POSITIVE    0.537
CONFIRMED         0.233
CANDIDATE         0.230
Name: proportion, dtype: float64

Overlapping stars: 0

Group-aware splitting successful!


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define preprocessing pipeline
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Learn preprocessing parameters from training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Apply learned transformations to test data
X_test_processed = preprocessor.transform(X_test)

# Validate results
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

print("\nMissing values after preprocessing:")
print("Training:", np.isnan(X_train_processed).sum())
print("Testing:", np.isnan(X_test_processed).sum())

print("\nTraining feature means (first 5):")
print(np.round(X_train_processed.mean(axis=0)[:5], 4))

print("\nTraining feature standard deviations (first 5):")
print(np.round(X_train_processed.std(axis=0)[:5], 4))

assert not np.isnan(X_train_processed).any()
assert not np.isnan(X_test_processed).any()

print("\nPreprocessing successful!")

Training shape: (7662, 37)
Testing shape: (1902, 37)

Missing values after preprocessing:
Training: 0
Testing: 0

Training feature means (first 5):
[ 0. -0. -0.  0.  0.]

Training feature standard deviations (first 5):
[1. 1. 1. 1. 1.]

Preprocessing successful!


## Model Training

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Build complete ML pipeline
lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Train using original training features
lr_pipeline.fit(X_train, y_train)

print("Logistic Regression training completed!")

print("\nClasses:")
print(lr_pipeline.classes_)

print("\nTraining accuracy:")
print(round(lr_pipeline.score(X_train, y_train), 4))

print("\nTesting accuracy:")
print(round(lr_pipeline.score(X_test, y_test), 4))

Logistic Regression training completed!

Classes:
['CANDIDATE' 'CONFIRMED' 'FALSE POSITIVE']

Training accuracy:
0.731

Testing accuracy:
0.725


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Generate predictions
y_pred_lr = lr_pipeline.predict(X_test)

# Generate class probabilities
y_proba_lr = lr_pipeline.predict_proba(X_test)

# Calculate six metrics
lr_metrics = {
    "Model": "Logistic Regression",

    "Accuracy": accuracy_score(
        y_test, y_pred_lr
    ),

    "AUC": roc_auc_score(
        y_test,
        y_proba_lr,
        multi_class="ovr",
        average="macro",
        labels=lr_pipeline.classes_
    ),

    "Precision": precision_score(
        y_test,
        y_pred_lr,
        average="macro",
        zero_division=0
    ),

    "Recall": recall_score(
        y_test,
        y_pred_lr,
        average="macro",
        zero_division=0
    ),

    "F1": f1_score(
        y_test,
        y_pred_lr,
        average="macro",
        zero_division=0
    ),

    "MCC": matthews_corrcoef(
        y_test,
        y_pred_lr
    )
}

# Display metrics
print("LOGISTIC REGRESSION PERFORMANCE")
print("=" * 40)

for metric, value in lr_metrics.items():
    if metric != "Model":
        print(f"{metric}: {value:.4f}")

# Detailed classification report
print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_test,
        y_pred_lr,
        zero_division=0
    )
)

# Confusion matrix
print("\nCONFUSION MATRIX")

cm = confusion_matrix(
    y_test,
    y_pred_lr,
    labels=lr_pipeline.classes_
)

print("Class order:", lr_pipeline.classes_)
print(cm)

LOGISTIC REGRESSION PERFORMANCE
Accuracy: 0.7250
AUC: 0.8598
Precision: 0.6581
Recall: 0.6651
F1: 0.6470
MCC: 0.5436

CLASSIFICATION REPORT
                precision    recall  f1-score   support

     CANDIDATE       0.50      0.28      0.36       438
     CONFIRMED       0.68      0.86      0.76       443
FALSE POSITIVE       0.80      0.86      0.83      1021

      accuracy                           0.73      1902
     macro avg       0.66      0.67      0.65      1902
  weighted avg       0.70      0.73      0.70      1902


CONFUSION MATRIX
Class order: ['CANDIDATE' 'CONFIRMED' 'FALSE POSITIVE']
[[121 132 185]
 [ 29 381  33]
 [ 93  51 877]]


## Train Decision Tree

In [20]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Build Decision Tree pipeline
dt_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ))
])

# Train
dt_pipeline.fit(X_train, y_train)

# Evaluate accuracy
dt_train_accuracy = dt_pipeline.score(X_train, y_train)
dt_test_accuracy = dt_pipeline.score(X_test, y_test)

print("DECISION TREE TRAINING COMPLETE")
print("=" * 40)

print(f"Training Accuracy: {dt_train_accuracy:.4f}")
print(f"Testing Accuracy:  {dt_test_accuracy:.4f}")
print(f"Accuracy Gap:      {dt_train_accuracy - dt_test_accuracy:.4f}")

DECISION TREE TRAINING COMPLETE
Training Accuracy: 0.8412
Testing Accuracy:  0.7382
Accuracy Gap:      0.1030


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import pandas as pd

def evaluate_model(model, X_test, y_test, model_name):

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(
            y_test,
            y_proba,
            multi_class="ovr",
            average="macro",
            labels=model.classes_
        ),
        "Precision": precision_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        ),
        "F1": f1_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        ),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

    print(f"\n{model_name.upper()}")
    print("=" * 45)

    display(pd.DataFrame([metrics]).round(4))

    print("\nClassification Report:")
    print(classification_report(
        y_test, y_pred, zero_division=0
    ))

    print("\nConfusion Matrix:")
    print("Class order:", model.classes_)
    print(confusion_matrix(
        y_test, y_pred,
        labels=model.classes_
    ))

    return metrics


# Evaluate Decision Tree
dt_metrics = evaluate_model(
    dt_pipeline,
    X_test,
    y_test,
    "Decision Tree"
)

# Store results for comparison
model_results = [lr_metrics, dt_metrics]

display(
    pd.DataFrame(model_results)
    .sort_values("F1", ascending=False)
    .round(4)
)


DECISION TREE


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Decision Tree,0.7382,0.8535,0.6947,0.7031,0.6985,0.5702



Classification Report:
                precision    recall  f1-score   support

     CANDIDATE       0.52      0.51      0.52       438
     CONFIRMED       0.72      0.78      0.75       443
FALSE POSITIVE       0.84      0.82      0.83      1021

      accuracy                           0.74      1902
     macro avg       0.69      0.70      0.70      1902
  weighted avg       0.74      0.74      0.74      1902


Confusion Matrix:
Class order: ['CANDIDATE' 'CONFIRMED' 'FALSE POSITIVE']
[[225  80 133]
 [ 71 345  27]
 [134  53 834]]


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
1,Decision Tree,0.7382,0.8535,0.6947,0.7031,0.6985,0.5702
0,Logistic Regression,0.7250,0.8598,0.6581,0.6651,0.6470,0.5436


## Train KNN

In [22]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

knn_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(
        n_neighbors=5,
        weights="distance",
        n_jobs=-1
    ))
])

# Train model
knn_pipeline.fit(X_train, y_train)

# Calculate accuracy
knn_train_accuracy = knn_pipeline.score(X_train, y_train)
knn_test_accuracy = knn_pipeline.score(X_test, y_test)

print("KNN TRAINING COMPLETE")
print("=" * 40)

print(f"Training Accuracy: {knn_train_accuracy:.4f}")
print(f"Testing Accuracy:  {knn_test_accuracy:.4f}")
print(
    f"Accuracy Gap:      "
    f"{knn_train_accuracy - knn_test_accuracy:.4f}"
)

KNN TRAINING COMPLETE
Training Accuracy: 1.0000
Testing Accuracy:  0.7103
Accuracy Gap:      0.2897


In [23]:
# Evaluate KNN
knn_metrics = evaluate_model(
    knn_pipeline,
    X_test,
    y_test,
    "KNN"
)

# Add KNN results
model_results.append(knn_metrics)

# Compare all trained models
comparison_df = pd.DataFrame(model_results)

display(
    comparison_df
    .sort_values("F1", ascending=False)
    .round(4)
)


KNN


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,KNN,0.7103,0.8306,0.6471,0.6596,0.6478,0.5216



Classification Report:
                precision    recall  f1-score   support

     CANDIDATE       0.47      0.35      0.40       438
     CONFIRMED       0.66      0.81      0.73       443
FALSE POSITIVE       0.81      0.82      0.82      1021

      accuracy                           0.71      1902
     macro avg       0.65      0.66      0.65      1902
  weighted avg       0.70      0.71      0.70      1902


Confusion Matrix:
Class order: ['CANDIDATE' 'CONFIRMED' 'FALSE POSITIVE']
[[152 123 163]
 [ 53 358  32]
 [119  61 841]]


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
1,Decision Tree,0.7382,0.8535,0.6947,0.7031,0.6985,0.5702
2,KNN,0.7103,0.8306,0.6471,0.6596,0.6478,0.5216
0,Logistic Regression,0.7250,0.8598,0.6581,0.6651,0.6470,0.5436


## Train Gaussian Naive Bayes

In [24]:
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Gaussian Naive Bayes pipeline
nb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", GaussianNB())
])

# Train
nb_pipeline.fit(X_train, y_train)

# Calculate accuracy
nb_train_accuracy = nb_pipeline.score(X_train, y_train)
nb_test_accuracy = nb_pipeline.score(X_test, y_test)

print("NAIVE BAYES TRAINING COMPLETE")
print("=" * 40)

print(f"Training Accuracy: {nb_train_accuracy:.4f}")
print(f"Testing Accuracy:  {nb_test_accuracy:.4f}")
print(
    f"Accuracy Gap:      "
    f"{nb_train_accuracy - nb_test_accuracy:.4f}"
)

NAIVE BAYES TRAINING COMPLETE
Training Accuracy: 0.4220
Testing Accuracy:  0.3927
Accuracy Gap:      0.0292


In [25]:
# Evaluate Naive Bayes
nb_metrics = evaluate_model(
    nb_pipeline,
    X_test,
    y_test,
    "Naive Bayes"
)

# Store results
model_results.append(nb_metrics)

# Compare models
comparison_df = (
    pd.DataFrame(model_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(comparison_df.round(4))


NAIVE BAYES


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Naive Bayes,0.3927,0.7142,0.4959,0.4648,0.3586,0.2149



Classification Report:
                precision    recall  f1-score   support

     CANDIDATE       0.17      0.18      0.18       438
     CONFIRMED       0.37      0.99      0.54       443
FALSE POSITIVE       0.95      0.23      0.36      1021

      accuracy                           0.39      1902
     macro avg       0.50      0.46      0.36      1902
  weighted avg       0.63      0.39      0.36      1902


Confusion Matrix:
Class order: ['CANDIDATE' 'CONFIRMED' 'FALSE POSITIVE']
[[ 80 347  11]
 [  5 437   1]
 [386 405 230]]


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Decision Tree,0.7382,0.8535,0.6947,0.7031,0.6985,0.5702
1,KNN,0.7103,0.8306,0.6471,0.6596,0.6478,0.5216
2,Logistic Regression,0.7250,0.8598,0.6581,0.6651,0.6470,0.5436
3,Naive Bayes,0.3927,0.7142,0.4959,0.4648,0.3586,0.2149


## Train SVM

In [26]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

svm_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        random_state=42
    ))
])

# Train SVM
svm_pipeline.fit(X_train, y_train)

# Evaluate accuracy
svm_train_accuracy = svm_pipeline.score(X_train, y_train)
svm_test_accuracy = svm_pipeline.score(X_test, y_test)

print("SVM TRAINING COMPLETE")
print("=" * 40)

print(f"Training Accuracy: {svm_train_accuracy:.4f}")
print(f"Testing Accuracy:  {svm_test_accuracy:.4f}")
print(
    f"Accuracy Gap:      "
    f"{svm_train_accuracy - svm_test_accuracy:.4f}"
)

SVM TRAINING COMPLETE
Training Accuracy: 0.7572
Testing Accuracy:  0.7371
Accuracy Gap:      0.0201


In [27]:
# Evaluate SVM
svm_metrics = evaluate_model(
    svm_pipeline,
    X_test,
    y_test,
    "SVM"
)

# Store results
model_results.append(svm_metrics)

# Compare all five models
comparison_df = (
    pd.DataFrame(model_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

print("\nFINAL BASELINE MODEL COMPARISON")
display(comparison_df.round(4))


SVM


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,SVM,0.7371,0.8755,0.6782,0.6696,0.6562,0.5611



Classification Report:
                precision    recall  f1-score   support

     CANDIDATE       0.55      0.28      0.37       438
     CONFIRMED       0.68      0.84      0.75       443
FALSE POSITIVE       0.80      0.89      0.84      1021

      accuracy                           0.74      1902
     macro avg       0.68      0.67      0.66      1902
  weighted avg       0.72      0.74      0.71      1902


Confusion Matrix:
Class order: ['CANDIDATE' 'CONFIRMED' 'FALSE POSITIVE']
[[124 128 186]
 [ 30 371  42]
 [ 71  43 907]]

FINAL BASELINE MODEL COMPARISON


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Decision Tree,0.7382,0.8535,0.6947,0.7031,0.6985,0.5702
1,SVM,0.7371,0.8755,0.6782,0.6696,0.6562,0.5611
2,KNN,0.7103,0.8306,0.6471,0.6596,0.6478,0.5216
3,Logistic Regression,0.7250,0.8598,0.6581,0.6651,0.6470,0.5436
4,Naive Bayes,0.3927,0.7142,0.4959,0.4648,0.3586,0.2149


## Cross-Validate All Five Models

In [28]:
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_validate
)

from sklearn.metrics import make_scorer
import pandas as pd
import numpy as np

# Define models
models = {
    "Logistic Regression": lr_pipeline,
    "Decision Tree": dt_pipeline,
    "KNN": knn_pipeline,
    "Naive Bayes": nb_pipeline,
    "SVM": svm_pipeline
}

# Group-aware cross-validation
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "Accuracy": "accuracy",
    "Precision": make_scorer(
        precision_score,
        average="macro",
        zero_division=0
    ),
    "Recall": make_scorer(
        recall_score,
        average="macro",
        zero_division=0
    ),
    "F1": make_scorer(
        f1_score,
        average="macro",
        zero_division=0
    ),
    "MCC": make_scorer(
        matthews_corrcoef
    )
}

# Store CV results
cv_results = []

for name, model in models.items():

    print(f"Cross-validating: {name}")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=train_groups,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise"
    )

    result = {"Model": name}

    for metric in scoring:
        result[f"{metric}_Mean"] = np.mean(
            scores[f"test_{metric}"]
        )
        result[f"{metric}_Std"] = np.std(
            scores[f"test_{metric}"]
        )

    cv_results.append(result)

# Display results
cv_df = pd.DataFrame(cv_results)

cv_df = cv_df.sort_values(
    "F1_Mean",
    ascending=False
).reset_index(drop=True)

print("\nGROUP-AWARE CROSS-VALIDATION RESULTS")

display(cv_df.round(4))

Cross-validating: Logistic Regression
Cross-validating: Decision Tree
Cross-validating: KNN
Cross-validating: Naive Bayes
Cross-validating: SVM

GROUP-AWARE CROSS-VALIDATION RESULTS


,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,MCC_Mean,MCC_Std
0,Decision Tree,0.7163,0.0138,0.6764,0.0147,0.6838,0.0155,0.6792,0.0143,0.5417,0.0211
1,SVM,0.7379,0.0072,0.6911,0.0028,0.6733,0.0064,0.6629,0.0113,0.5679,0.0071
2,Logistic Regression,0.7251,0.0107,0.6668,0.0102,0.6653,0.0077,0.6510,0.0101,0.5473,0.0163
3,KNN,0.6951,0.0128,0.6348,0.0134,0.6452,0.0142,0.6344,0.0150,0.5016,0.0206
4,Naive Bayes,0.4227,0.0110,0.5151,0.0101,0.4888,0.0094,0.3949,0.0089,0.2455,0.0175


## Export Models

In [29]:
import joblib
import os
import json

# Create output directory
os.makedirs("models", exist_ok=True)

# Save all five pipelines
model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "SVM": "svm.joblib"
}

for model_name, filename in model_files.items():

    path = os.path.join("models", filename)

    joblib.dump(models[model_name], path)

    print(f"Saved: {path}")

# Save expected feature names
feature_names = X_train.columns.tolist()

with open("models/feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print("\nTotal features:", len(feature_names))
print("All models exported successfully!")

Saved: models/logistic_regression.joblib
Saved: models/decision_tree.joblib
Saved: models/knn.joblib
Saved: models/naive_bayes.joblib
Saved: models/svm.joblib

Total features: 37
All models exported successfully!


## Export Test CSV

In [30]:
# Combine test features with actual labels
test_export = X_test.copy()

test_export["koi_disposition"] = y_test.values

# Export CSV
test_export.to_csv(
    "test_data.csv",
    index=False
)

print("Test dataset exported!")

print("Shape:", test_export.shape)

print("\nTarget distribution:")
print(test_export["koi_disposition"].value_counts())

print("\nPreview:")
display(test_export.head())

Test dataset exported!
Shape: (1902, 38)

Target distribution:
koi_disposition
FALSE POSITIVE    1021
CONFIRMED          443
CANDIDATE          438
Name: count, dtype: int64

Preview:


,rowid,koi_period,koi_period_err1,koi_period_err2,koi_time0bk,koi_time0bk_err1,koi_time0bk_err2,koi_impact,koi_impact_err1,koi_impact_err2,...,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag,koi_disposition
5,6,11.094321,0.000020,-0.000020,171.201160,0.001410,-0.001410,0.538,0.030,-0.428,...,4.486,0.054,-0.229,0.972,0.315,-0.105,296.28613,48.224670,15.714,CONFIRMED
6,7,4.134435,0.000010,-0.000010,172.979370,0.001900,-0.001900,0.762,0.139,-0.532,...,4.486,0.054,-0.229,0.972,0.315,-0.105,296.28613,48.224670,15.714,CONFIRMED
7,8,2.566589,0.000018,-0.000018,179.554370,0.004610,-0.004610,0.755,0.212,-0.523,...,4.486,0.054,-0.229,0.972,0.315,-0.105,296.28613,48.224670,15.714,CONFIRMED
13,14,3.709214,0.000007,-0.000007,133.983180,0.001430,-0.001430,0.051,0.395,-0.051,...,4.407,0.085,-0.114,1.022,0.143,-0.107,295.64871,48.495560,12.772,CONFIRMED
15,16,19.403938,0.000021,-0.000021,172.484253,0.000842,-0.000842,0.804,0.007,-0.005,...,4.591,0.072,-0.048,0.680,0.072,-0.065,289.25821,47.635319,15.487,FALSE POSITIVE


## Download model Artifacts

In [ ]:
import joblib
import json
from pathlib import Path

# Create folders
Path("../models").mkdir(exist_ok=True)
Path("../data").mkdir(exist_ok=True)

# Save trained models
for name, filename in {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "SVM": "svm.joblib"
}.items():
    joblib.dump(models[name], f"../models/{filename}")

# Save feature names
with open("../models/feature_names.json", "w") as f:
    json.dump(X_train.columns.tolist(), f)

# Save test dataset
X_test.assign(
    koi_disposition=y_test
).to_csv("../data/test_data.csv", index=False)

print("Models and test data saved successfully!")